# Latin Hypercube sampling hyperparameters

**Benchamarked Models (regression problem)**

- Explainable Boosting (GAM ensemble boosting)
https://interpret.ml/docs/python/api/ExplainableBoostingRegressor.html

- Random Forest (ensemble bagging)
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

- Extreme Gradient Boosting (ensemble boosting)
https://xgboost.readthedocs.io/en/stable/parameter.html 

- Super Vector (kernel)
https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html#sklearn.svm.SVR

- Elastic Net (regularized linear)
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html#sklearn.linear_model.ElasticNet

- Deep Learning (neural networks)
https://docs.fast.ai/


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
seed=112358

The optimization of the hyperparameters of many ML models is challenging due to the tightly coupled hyperparameter space. We use a LH sampling to get more robust results about the models' performance


In [4]:
# Generate Latin Hypercube Sampling (LHS) columns (mid-quantiles per parameter).
# Each column has Nlhs stratified values (mid-quantiles) for the given scale.
# It's necessary to permute the columns after to ensure a Latin hypercube structure.
def generate_lhs_parameters(hyperparameters, Nlhs):
    lhs_pam = {}
    for param, config in hyperparameters.items():
        scale, dtype, a, b = config[:4]

        if scale == 'lin':
            # mid-quantiles on [a, b]
            am = a + (b - a) / (2 * Nlhs)
            bm = b - (b - a) / (2 * Nlhs)
            vals = np.linspace(am, bm, Nlhs)

        elif scale in ('log-uniform'):
            # mid-quantiles in log space (requires a,b > 0; you said you check elsewhere)
            log_a, log_b = np.log(a), np.log(b)
            log_am = log_a + (log_b - log_a) / (2 * Nlhs)
            log_bm = log_b - (log_b - log_a) / (2 * Nlhs)
            vals = np.exp(np.linspace(log_am, log_bm, Nlhs))

        elif scale == 'const':
            # constant value a
            vals = np.full(Nlhs, a, dtype=float)

        else:
            raise ValueError(f"Unknown scale type '{scale}' for parameter '{param}'")

        lhs_pam[param] = np.rint(vals).astype(int) if dtype == 'int' else vals.astype(float)
    
    return lhs_pam


In [5]:
hyperparameters=dict() # Dictionary to store the hyperparameters

In [6]:
# Random Forest hyperparameters
hyperparameters['rf'] = {'n_estimators':('lin','int',100,800),
                   'max_depth':('lin','int',3,30),
                   'min_samples_leaf':('log-uniform','int',1,50),
                   'max_features':('lin','float',0.3,1)}

In [7]:
# XGBoost hyperparameters
hyperparameters['xgb'] = {'n_estimators':('lin','int',100,1000),# model complexity
                   'learning_rate':('log-uniform','float',0.01,0.3), # model complexity
                   'max_depth':('lin','int',2,10), # model complexiity
                   'subsample':('lin','float',0.5,1.0), # randomness 
                   'colsample_bytree':('lin','float',0.5,1.0), # randomness
                   'min_child_weight':('log-uniform','float',1,20), # model complexity

                   'reg_alpha':('log-uniform','float',0.01,1), # model complexity (L1 regularization term)
                   'reg_lambda':('log-uniform','float',0.01,10), # model complexity (L2 regularization term)
                   } 
# gamma=0 (default value)


In [8]:
# EBM hyperparameters
hyperparameters['ebm'] = {'interactions':('lin','float',0,0.9),
                   'learning_rate':('log-uniform','float',0.005,0.2),
                   'max_leaves':('lin','int',2,4),
                   'min_samples_leaf':('log-uniform','int',1,100),
                   'early_stopping_rounds':('lin','int',50,100)
}

In [9]:
# Super Vector Machine (SVM) hyperparameters
hyperparameters['sv'] = {'C':('log-uniform','float',0.1,100), # regularization parameter
                'epsilon':('lin','float',0.01,0.5), # epsilon in the epsilon-SVR           
#                   'gamma':('log-uniform','float',1e-4,1e2) # 10^-4 to 10^2 Kernel coefficient for ‘rbf’
}
# gamma='scale' (default value); kernel='rbf'

In [10]:
# Elastic Net hyperparameters
# (add interaction terms to the model, i.e. PolynomialFeatures(2, interaction_only=True, include_bias=False))
hyperparameters['en'] = {'alpha':('log-uniform','float',1e-4,1),
                   'l1_ratio':('lin','float',0,1)}

In [11]:
# FastAI hyperparameters
hyperparameters['fastai'] = {
    'lr_max': ('log-uniform','float',1e-5,1e-1), # Maximum learning rate    
    'layer1': ('lin','int',100,500), # Number of neurons in the first layer
    'layer2': ('lin','int',50,250), # Number of neurons in the second layer
    'wd': ('log-uniform','float',1e-4,1e-2) # Weight decay (10^-4 to 10^-2)
}
# Learning rate is optimized in each trial by lr_find function
# epochs is set to 100 but it is controlled by early stopping

In [12]:
hyperparameters.keys()

dict_keys(['rf', 'xgb', 'ebm', 'sv', 'en', 'fastai'])

In [15]:
Nlhs=50 # Number of combinations of parameters for LHS

In [17]:
for model in hyperparameters.keys():
    #  Generate Latin Hypercube Sampling (LHS) parameters
    df_pam=pd.DataFrame(generate_lhs_parameters(hyperparameters[model],Nlhs))
    df_pam.index.name = 'lhs'
    # Random permutation of the columns
    rng = np.random.default_rng(seed)
    for c in df_pam.columns:
        df_pam[c] = rng.permutation(df_pam[c].to_numpy())
    #df_pam=df_pam.apply(lambda column: np.random.permutation(column), axis=0)
    # Save the parameters to a CSV file
    df_pam.to_csv(f"./model_selection/hyperparameters/lhs_{model}_parameters.csv",index=True)
